In [1]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "ubuntu_log_auditor_rag"
LOG_TABLE_ID = "ubuntu_log_files"
DOCUMENT_TABLE_ID = "ubuntu_knowledge_documents"
CHUNK_TABLE_ID = "ubuntu_log_and_knowledge_chunks"
AUDIT_TABLE_ID = "ubuntu_audit_reports"

PLANNING_MODEL = "gemini-2.5-pro"
TEXT_MODEL = "gemini-2.5-flash"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CHUNK_SIZE_CHARS = 1200
CHUNK_OVERLAP_CHARS = 200
MAX_EMBEDDING_TEXT_CHARS = 3000

GCS_LOG_PREFIX = "ubuntu-log-auditor/logs"
GCS_AUDIT_PREFIX = "ubuntu-log-auditor/audits"
GCS_SCRIPT_PREFIX = "ubuntu-log-auditor/scripts"
GCS_SUMMARY_PREFIX = "ubuntu-log-auditor/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Dataset:", DATASET_ID)
print("Text model:", TEXT_MODEL)
print("Embedding model:", TEXT_EMBEDDING_MODEL)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Dataset: ubuntu_log_auditor_rag
Text model: gemini-2.5-flash
Embedding model: text-embedding-005


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("BigQuery project:", bigquery_client.project)
print("Text embedding model loaded.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
BigQuery project: leafy-guide-497515-m4
Text embedding model loaded.


In [4]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"

try:
    dataset = bigquery_client.create_dataset(dataset_ref)
    print("Created dataset:", dataset.full_dataset_id)
except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)
    print("Dataset already exists:", dataset.full_dataset_id)

log_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{LOG_TABLE_ID}"
document_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{DOCUMENT_TABLE_ID}"
chunk_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
audit_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{AUDIT_TABLE_ID}"

print("Log table:", log_table_ref)
print("Document table:", document_table_ref)
print("Chunk table:", chunk_table_ref)
print("Audit table:", audit_table_ref)

Created dataset: leafy-guide-497515-m4:ubuntu_log_auditor_rag
Log table: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_files
Document table: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_knowledge_documents
Chunk table: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_and_knowledge_chunks
Audit table: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_audit_reports


In [5]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)
        print("Table already exists:", table.full_table_id)
        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)
        print("Created table:", created_table.full_table_id)

        return created_table

In [6]:
log_schema = [
    bigquery.SchemaField("log_file_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("log_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("log_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("host_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("scenario", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("severity_hint", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("gcs_uri", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

document_schema = [
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("document_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("topic", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

chunk_schema = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_kind", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("global_chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("topic", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_char_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("embedding_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

audit_schema = [
    bigquery.SchemaField("audit_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("risk_level", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("primary_issue", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("root_cause_hypothesis", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("audit_report", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("bash_script", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("audit_gcs_uri", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("script_gcs_uri", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("used_chunk_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("used_sources", "STRING", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

In [7]:
log_table = ensure_bigquery_table(
    log_table_ref,
    log_schema,
)

document_table = ensure_bigquery_table(
    document_table_ref,
    document_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

audit_table = ensure_bigquery_table(
    audit_table_ref,
    audit_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_files
Created table: leafy-guide-497515-m4:ubuntu_log_auditor_rag.ubuntu_log_files
Table does not exist. Creating: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_knowledge_documents
Created table: leafy-guide-497515-m4:ubuntu_log_auditor_rag.ubuntu_knowledge_documents
Table does not exist. Creating: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_and_knowledge_chunks
Created table: leafy-guide-497515-m4:ubuntu_log_auditor_rag.ubuntu_log_and_knowledge_chunks
Table does not exist. Creating: leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_audit_reports
Created table: leafy-guide-497515-m4:ubuntu_log_auditor_rag.ubuntu_audit_reports


In [8]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str = "text/plain",
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )

In [9]:
def extract_json_object(text: str) -> dict:
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```").strip()

    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in text: {text[:500]}")

    return json.loads(cleaned[start:end + 1])

In [10]:
ubuntu_auditor_brief = """
Create a synthetic Ubuntu system log auditing dataset.

The notebook should simulate a cloud-native log auditor for Ubuntu servers.
It should not read local /var/log files and should not save local files.
All generated logs, audit reports, scripts, and metadata must be stored in Google Cloud Storage and BigQuery.

The system should analyze:
- syslog-like system events
- dmesg-like kernel messages
- systemd service failures
- disk pressure warnings
- SSH authentication anomalies
- network instability
- package manager issues
- kernel driver warnings

The final assistant should:
- retrieve relevant log chunks and knowledge chunks semantically
- identify likely root cause
- rate operational risk
- propose safe diagnostic commands
- generate a cautious Bash mitigation script
- explicitly avoid destructive commands unless clearly justified
"""

print(ubuntu_auditor_brief)


Create a synthetic Ubuntu system log auditing dataset.

The notebook should simulate a cloud-native log auditor for Ubuntu servers.
It should not read local /var/log files and should not save local files.
All generated logs, audit reports, scripts, and metadata must be stored in Google Cloud Storage and BigQuery.

The system should analyze:
- syslog-like system events
- dmesg-like kernel messages
- systemd service failures
- disk pressure warnings
- SSH authentication anomalies
- network instability
- package manager issues
- kernel driver warnings

The final assistant should:
- retrieve relevant log chunks and knowledge chunks semantically
- identify likely root cause
- rate operational risk
- propose safe diagnostic commands
- generate a cautious Bash mitigation script
- explicitly avoid destructive commands unless clearly justified



In [11]:
log_specs = [
    {
        "log_number": 1,
        "log_type": "syslog",
        "host_name": "ubuntu-worker-01",
        "scenario": "disk pressure and journal growth causing service instability",
        "severity_hint": "high",
        "focus": "systemd journal growth, low disk space, apt cache, service restart loops",
    },
    {
        "log_number": 2,
        "log_type": "dmesg",
        "host_name": "ubuntu-gpu-node-02",
        "scenario": "kernel driver warnings and intermittent device reset",
        "severity_hint": "medium",
        "focus": "kernel ring buffer, PCIe device reset, driver timeout, I/O warning",
    },
    {
        "log_number": 3,
        "log_type": "authlog",
        "host_name": "ubuntu-api-03",
        "scenario": "SSH authentication anomaly with repeated failed login attempts",
        "severity_hint": "critical",
        "focus": "failed password attempts, invalid users, sudo session, accepted public key, suspicious source IP patterns",
    },
    {
        "log_number": 4,
        "log_type": "systemd",
        "host_name": "ubuntu-db-04",
        "scenario": "database service restart loop after configuration change",
        "severity_hint": "high",
        "focus": "systemd restart policy, failed unit, environment file mismatch, port binding conflict",
    },
    {
        "log_number": 5,
        "log_type": "network",
        "host_name": "ubuntu-edge-05",
        "scenario": "network instability with DNS failures and interface flaps",
        "severity_hint": "medium",
        "focus": "NetworkManager, DNS timeout, link down/up, packet loss, route changes",
    },
]

pd.DataFrame(log_specs)

,log_number,log_type,host_name,scenario,severity_hint,focus
0,1,syslog,ubuntu-worker-01,disk pressure and journal growth causing servi...,high,"systemd journal growth, low disk space, apt ca..."
1,2,dmesg,ubuntu-gpu-node-02,kernel driver warnings and intermittent device...,medium,"kernel ring buffer, PCIe device reset, driver ..."
2,3,authlog,ubuntu-api-03,SSH authentication anomaly with repeated faile...,critical,"failed password attempts, invalid users, sudo ..."
3,4,systemd,ubuntu-db-04,database service restart loop after configurat...,high,"systemd restart policy, failed unit, environme..."
4,5,network,ubuntu-edge-05,network instability with DNS failures and inte...,medium,"NetworkManager, DNS timeout, link down/up, pac..."


In [12]:
def generate_synthetic_ubuntu_log(spec: dict) -> str:
    prompt = f"""
You are generating a synthetic Ubuntu log file for a cloud log auditing lab.

Brief:
{ubuntu_auditor_brief}

Log specification:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Create a realistic synthetic Linux log file.

Requirements:
- Use realistic Ubuntu log style.
- Include timestamps, hostnames, process names, PIDs where appropriate.
- Include 80 to 140 log lines.
- Include normal noise and relevant warning/error lines.
- Make the issue diagnosable but not too obvious.
- Do not include real secrets, real IP addresses, or real user data.
- Use documentation-safe private IP ranges only, such as 10.x.x.x, 172.16.x.x, 192.168.x.x.
- Do not mention that the log is synthetic.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()

In [13]:
generated_logs = []

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

for spec in log_specs:
    print("=" * 100)
    print("Generating log:", spec["log_type"], "|", spec["host_name"])

    content = generate_synthetic_ubuntu_log(spec)

    blob_name = (
        f"{GCS_LOG_PREFIX}/"
        f"{timestamp}/"
        f"{spec['log_number']:02d}_{spec['host_name']}_{spec['log_type']}.log"
    )

    gcs_uri = upload_text_to_gcs(
        content,
        blob_name=blob_name,
        content_type="text/plain",
    )

    log_row = {
        "log_file_id": str(uuid4()),
        "log_number": spec["log_number"],
        "log_type": spec["log_type"],
        "host_name": spec["host_name"],
        "scenario": spec["scenario"],
        "severity_hint": spec["severity_hint"],
        "content": content,
        "gcs_uri": gcs_uri,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    generated_logs.append(log_row)

    print("Lines:", len(content.splitlines()))
    print("GCS URI:", gcs_uri)
    print(content[:800])

Generating log: syslog | ubuntu-worker-01
Lines: 103
GCS URI: gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/logs/20260702_151254/01_ubuntu-worker-01_syslog.log
Oct 26 08:30:01 ubuntu-worker-01 systemd[1]: Starting Daily apt download activities...
Oct 26 08:30:02 ubuntu-worker-01 CRON[23101]: (root) CMD (   PATH="$PATH:/usr/sbin"  cd / && run-parts --report /etc/cron.hourly)
Oct 26 08:30:15 ubuntu-worker-01 sshd[23115]: Accepted publickey for cloud-user from 10.0.1.15 port 56284 ssh2: RSA SHA256:xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
Oct 26 08:30:15 ubuntu-worker-01 systemd[1]: Created slice User Slice of UID 1000.
Oct 26 08:30:15 ubuntu-worker-01 systemd[1]: Starting User Runtime Directory /run/user/1000...
Oct 26 08:30:16 ubuntu-worker-01 systemd[1]: Started User Runtime Directory /run/user/1000.
Oct 26 08:30:16 ubuntu-worker-01 systemd[1]: Starting User Manager for UID 1000...
Oct 26 08:31:01 ubuntu-worker-01 systemd[1]: Starting Clean Up 
Generating log: dmesg | u

In [14]:
knowledge_specs = [
    {
        "document_number": 1,
        "document_type": "runbook",
        "topic": "disk_pressure",
        "title": "Ubuntu Disk Pressure and Journal Cleanup Runbook",
        "focus": "journalctl disk usage, apt cache cleanup, log rotation, safe disk investigation",
    },
    {
        "document_number": 2,
        "document_type": "runbook",
        "topic": "systemd_failures",
        "title": "Systemd Service Failure and Restart Loop Runbook",
        "focus": "systemctl status, journalctl -u, restart policy, config validation, rollback",
    },
    {
        "document_number": 3,
        "document_type": "runbook",
        "topic": "ssh_auth_anomaly",
        "title": "SSH Authentication Anomaly Investigation Runbook",
        "focus": "failed login attempts, auth.log inspection, sshd config, fail2ban, account review",
    },
    {
        "document_number": 4,
        "document_type": "runbook",
        "topic": "kernel_driver_warnings",
        "title": "Kernel Driver Warning and Device Reset Runbook",
        "focus": "dmesg analysis, driver timeouts, PCIe reset, kernel module checks, firmware",
    },
    {
        "document_number": 5,
        "document_type": "runbook",
        "topic": "network_instability",
        "title": "Ubuntu Network Instability and DNS Troubleshooting Runbook",
        "focus": "NetworkManager logs, DNS resolution, route inspection, packet loss, interface flaps",
    },
]

pd.DataFrame(knowledge_specs)

,document_number,document_type,topic,title,focus
0,1,runbook,disk_pressure,Ubuntu Disk Pressure and Journal Cleanup Runbook,"journalctl disk usage, apt cache cleanup, log ..."
1,2,runbook,systemd_failures,Systemd Service Failure and Restart Loop Runbook,"systemctl status, journalctl -u, restart polic..."
2,3,runbook,ssh_auth_anomaly,SSH Authentication Anomaly Investigation Runbook,"failed login attempts, auth.log inspection, ss..."
3,4,runbook,kernel_driver_warnings,Kernel Driver Warning and Device Reset Runbook,"dmesg analysis, driver timeouts, PCIe reset, k..."
4,5,runbook,network_instability,Ubuntu Network Instability and DNS Troubleshoo...,"NetworkManager logs, DNS resolution, route ins..."


In [15]:
def generate_knowledge_document(spec: dict) -> str:
    prompt = f"""
You are a senior Linux SRE writing internal Ubuntu troubleshooting documentation.

Brief:
{ubuntu_auditor_brief}

Document specification:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Write a realistic internal runbook in English with these sections:
1. Problem overview
2. Common log patterns
3. First diagnostic commands
4. Risk assessment
5. Safe mitigation options
6. Commands to avoid unless approved
7. Escalation criteria
8. Example Bash helper script

Requirements:
- Keep it practical for Ubuntu.
- Include concrete command examples.
- Do not include destructive commands like rm -rf.
- Prefer safe inspection commands first.
- Include enough text to split into chunks.
- Do not mention that this is AI-generated.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()

In [16]:
knowledge_documents = []

for spec in knowledge_specs:
    print("=" * 100)
    print("Generating knowledge document:", spec["title"])

    content = generate_knowledge_document(spec)

    document = {
        "document_id": str(uuid4()),
        "document_number": spec["document_number"],
        "document_type": spec["document_type"],
        "topic": spec["topic"],
        "title": spec["title"],
        "content": content,
        "summary": content[:700],
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    knowledge_documents.append(document)

    print("Characters:", len(content))
    print(content[:700])

Generating knowledge document: Ubuntu Disk Pressure and Journal Cleanup Runbook
Characters: 11062
```json
{
  "document_number": 1,
  "document_type": "runbook",
  "topic": "disk_pressure",
  "title": "Ubuntu Disk Pressure and Journal Cleanup Runbook",
  "focus": "journalctl disk usage, apt cache cleanup, log rotation, safe disk investigation"
}
```

---

### **1. Problem Overview**

This runbook addresses alerts related to disk pressure on Ubuntu servers, with a specific focus on issues originating from log accumulation, particularly `systemd-journald`. When a filesystem, especially the root partition (`/`) or `/var`, approaches full capacity, system stability is compromised.

Symptoms manifest as:
*   Application failures with "No space left on device" errors.
*   Inability to write ne
Generating knowledge document: Systemd Service Failure and Restart Loop Runbook
Characters: 12626
```json
{
  "document_number": 2,
  "document_type": "runbook",
  "topic": "systemd_failures",
  "title

In [17]:
logs_df = pd.DataFrame(
    [
        {
            "log_number": log["log_number"],
            "log_type": log["log_type"],
            "host_name": log["host_name"],
            "scenario": log["scenario"],
            "severity_hint": log["severity_hint"],
            "line_count": len(log["content"].splitlines()),
            "gcs_uri": log["gcs_uri"],
        }
        for log in generated_logs
    ]
)

documents_df = pd.DataFrame(
    [
        {
            "document_number": doc["document_number"],
            "document_type": doc["document_type"],
            "topic": doc["topic"],
            "title": doc["title"],
            "content_length": len(doc["content"]),
        }
        for doc in knowledge_documents
    ]
)

display(logs_df)
display(documents_df)

,log_number,log_type,host_name,scenario,severity_hint,line_count,gcs_uri
0,1,syslog,ubuntu-worker-01,disk pressure and journal growth causing servi...,high,103,gs://leafy-guide-497515-m4-vector-assets/ubunt...
1,2,dmesg,ubuntu-gpu-node-02,kernel driver warnings and intermittent device...,medium,124,gs://leafy-guide-497515-m4-vector-assets/ubunt...
2,3,authlog,ubuntu-api-03,SSH authentication anomaly with repeated faile...,critical,99,gs://leafy-guide-497515-m4-vector-assets/ubunt...
3,4,systemd,ubuntu-db-04,database service restart loop after configurat...,high,92,gs://leafy-guide-497515-m4-vector-assets/ubunt...
4,5,network,ubuntu-edge-05,network instability with DNS failures and inte...,medium,107,gs://leafy-guide-497515-m4-vector-assets/ubunt...


,document_number,document_type,topic,title,content_length
0,1,runbook,disk_pressure,Ubuntu Disk Pressure and Journal Cleanup Runbook,11062
1,2,runbook,systemd_failures,Systemd Service Failure and Restart Loop Runbook,12626
2,3,runbook,ssh_auth_anomaly,SSH Authentication Anomaly Investigation Runbook,13957
3,4,runbook,kernel_driver_warnings,Kernel Driver Warning and Device Reset Runbook,11716
4,5,runbook,network_instability,Ubuntu Network Instability and DNS Troubleshoo...,10490


In [18]:
def normalize_text(text: str) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(start + chunk_size_chars, len(clean_text))

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(". ", start, end)

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(0, end - chunk_overlap_chars)

    return chunks

In [19]:
def normalize_text(text: str) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(start + chunk_size_chars, len(clean_text))

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(". ", start, end)

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(0, end - chunk_overlap_chars)

    return chunks

In [20]:
def shorten_text_for_embedding(
    text: str,
    *,
    max_chars: int = MAX_EMBEDDING_TEXT_CHARS,
) -> str:
    clean_text = normalize_text(text)

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_text_embedding(text: str) -> list[float]:
    safe_text = shorten_text_for_embedding(text)

    embeddings = text_embedding_model.get_embeddings(
        [safe_text]
    )

    return list(embeddings[0].values)

In [22]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for log in generated_logs:
    chunks = split_text_into_chunks(log["content"])

    print("=" * 100)
    print("Log:", log["host_name"], log["log_type"])
    print("Chunks:", len(chunks))

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "source_id": log["log_file_id"],
                "source_kind": "log_file",
                "source_number": log["log_number"],
                "chunk_number": chunk_index,
                "global_chunk_number": global_chunk_number,
                "source_type": log["log_type"],
                "topic": log["scenario"],
                "title": f"{log['host_name']} {log['log_type']} log",
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        global_chunk_number += 1

for document in knowledge_documents:
    chunks = split_text_into_chunks(document["content"])

    print("=" * 100)
    print("Document:", document["title"])
    print("Chunks:", len(chunks))

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "source_id": document["document_id"],
                "source_kind": "knowledge_document",
                "source_number": document["document_number"],
                "chunk_number": chunk_index,
                "global_chunk_number": global_chunk_number,
                "source_type": document["document_type"],
                "topic": document["topic"],
                "title": document["title"],
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        global_chunk_number += 1

print("Total chunks:", len(chunk_rows_without_embeddings))

chunks_preview_df = pd.DataFrame(
    [
        {
            "global_chunk_number": row["global_chunk_number"],
            "source_kind": row["source_kind"],
            "source_type": row["source_type"],
            "topic": row["topic"],
            "chunk_char_count": row["chunk_char_count"],
            "preview": row["chunk_text"][:200],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(12)

Log: ubuntu-worker-01 syslog
Chunks: 15
Log: ubuntu-gpu-node-02 dmesg
Chunks: 10
Log: ubuntu-api-03 authlog
Chunks: 11
Log: ubuntu-db-04 systemd
Chunks: 10
Log: ubuntu-edge-05 network
Chunks: 14
Document: Ubuntu Disk Pressure and Journal Cleanup Runbook
Chunks: 12
Document: Systemd Service Failure and Restart Loop Runbook
Chunks: 14
Document: SSH Authentication Anomaly Investigation Runbook
Chunks: 16
Document: Kernel Driver Warning and Device Reset Runbook
Chunks: 13
Document: Ubuntu Network Instability and DNS Troubleshooting Runbook
Chunks: 11
Total chunks: 126


,global_chunk_number,source_kind,source_type,topic,chunk_char_count,preview
0,1,log_file,syslog,disk pressure and journal growth causing servi...,1191,Oct 26 08:30:01 ubuntu-worker-01 systemd[1]: S...
1,2,log_file,syslog,disk pressure and journal growth causing servi...,1127,32:41 ubuntu-worker-01 systemd[1]: kubelet.ser...
2,3,log_file,syslog,disk pressure and journal growth causing servi...,1103,sult 'exit-code'. Oct 26 08:33:57 ubuntu-worke...
3,4,log_file,syslog,disk pressure and journal growth causing servi...,823,"ask_timeout_secs"" disables this message. Oct 2..."
4,5,log_file,syslog,disk pressure and journal growth causing servi...,879,"ce: Scheduled restart job, restart counter is ..."
5,6,log_file,syslog,disk pressure and journal growth causing servi...,1013,"ce: Scheduled restart job, restart counter is ..."
6,7,log_file,syslog,disk pressure and journal growth causing servi...,1164,"ce: Scheduled restart job, restart counter is ..."
7,8,log_file,syslog,disk pressure and journal growth causing servi...,771,orker-01 systemd[1]: apt-daily-upgrade.service...
8,9,log_file,syslog,disk pressure and journal growth causing servi...,1200,"ce: Scheduled restart job, restart counter is ..."
9,10,log_file,syslog,disk pressure and journal growth causing servi...,865,39:01 ubuntu-worker-01 CRON[23620]: pam_unix(c...


In [23]:
chunk_rows = []

for row in chunk_rows_without_embeddings:
    print(
        "Embedding chunk:",
        row["global_chunk_number"],
        "|",
        row["source_kind"],
        "|",
        row["source_type"],
    )

    embedding = get_text_embedding(row["chunk_text"])

    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

    print("Embedding dim:", len(embedding))
    print("Chunk chars:", row["chunk_char_count"])
    print("-" * 100)

print("Chunk rows with embeddings:", len(chunk_rows))

Embedding chunk: 1 | log_file | syslog
Embedding dim: 768
Chunk chars: 1191
----------------------------------------------------------------------------------------------------
Embedding chunk: 2 | log_file | syslog
Embedding dim: 768
Chunk chars: 1127
----------------------------------------------------------------------------------------------------
Embedding chunk: 3 | log_file | syslog
Embedding dim: 768
Chunk chars: 1103
----------------------------------------------------------------------------------------------------
Embedding chunk: 4 | log_file | syslog
Embedding dim: 768
Chunk chars: 823
----------------------------------------------------------------------------------------------------
Embedding chunk: 5 | log_file | syslog
Embedding dim: 768
Chunk chars: 879
----------------------------------------------------------------------------------------------------
Embedding chunk: 6 | log_file | syslog
Embedding dim: 768
Chunk chars: 1013
-----------------------------------------

In [24]:
log_rows = []

for log in generated_logs:
    log_rows.append(
        {
            "log_file_id": log["log_file_id"],
            "log_number": log["log_number"],
            "log_type": log["log_type"],
            "host_name": log["host_name"],
            "scenario": log["scenario"],
            "severity_hint": log["severity_hint"],
            "content": log["content"],
            "gcs_uri": log["gcs_uri"],
            "created_at": log["created_at"],
        }
    )

document_rows = []

for document in knowledge_documents:
    document_rows.append(
        {
            "document_id": document["document_id"],
            "document_number": document["document_number"],
            "document_type": document["document_type"],
            "topic": document["topic"],
            "title": document["title"],
            "content": document["content"],
            "summary": document["summary"],
            "created_at": document["created_at"],
        }
    )

bigquery_client.query(
    f"TRUNCATE TABLE `{log_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{document_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{chunk_table_ref}`",
    location=dataset.location,
).result()

log_errors = bigquery_client.insert_rows_json(
    log_table_ref,
    log_rows,
)

document_errors = bigquery_client.insert_rows_json(
    document_table_ref,
    document_rows,
)

chunk_errors = bigquery_client.insert_rows_json(
    chunk_table_ref,
    chunk_rows,
)

if log_errors:
    print("Log insert errors:")
    print(log_errors)
else:
    print("Inserted log rows:", len(log_rows))

if document_errors:
    print("Document insert errors:")
    print(document_errors)
else:
    print("Inserted document rows:", len(document_rows))

if chunk_errors:
    print("Chunk insert errors:")
    print(chunk_errors)
else:
    print("Inserted chunk rows:", len(chunk_rows))

Inserted log rows: 5
Inserted document rows: 5
Inserted chunk rows: 126


In [25]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{log_table_ref}`) AS log_count,
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Log count:", result.log_count)
print("Document count:", result.document_count)
print("Chunk count:", result.chunk_count)
print("Embedding length:", result.embedding_length)

Log count: 5
Document count: 5
Chunk count: 126
Embedding length: 768


In [26]:
def search_ubuntu_chunks(
    query: str,
    *,
    top_k: int = 10,
    source_kind: str | None = None,
    source_type: str | None = None,
) -> list[dict]:
    query_embedding = get_text_embedding(query)

    filters = []

    query_parameters = [
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding,
        ),
        bigquery.ScalarQueryParameter(
            "top_k",
            "INT64",
            top_k,
        ),
    ]

    if source_kind is not None:
        filters.append("source_kind = @source_kind")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "source_kind",
                "STRING",
                source_kind,
            )
        )

    if source_type is not None:
        filters.append("source_type = @source_type")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "source_type",
                "STRING",
                source_type,
            )
        )

    where_sql = ""

    if filters:
        where_sql = "WHERE " + " AND ".join(filters)

    sql = f"""
    SELECT
      base.chunk_id,
      base.source_id,
      base.source_kind,
      base.source_number,
      base.chunk_number,
      base.global_chunk_number,
      base.source_type,
      base.topic,
      base.title,
      base.chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
        {where_sql}
      ),
      'embedding',
      (SELECT @query_embedding AS embedding),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=query_parameters,
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "source_id": row.source_id,
            "source_kind": row.source_kind,
            "source_number": row.source_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "topic": row.topic,
            "title": row.title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [27]:
query = """
Ubuntu server has low disk space, systemd services are restarting,
journal logs are growing, and apt cache may be consuming space.
What should be inspected first?
"""

search_results = search_ubuntu_chunks(
    query,
    top_k=10,
)

print("Query:")
print(query)

print("\nSearch results:", len(search_results))

for result in search_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Source kind:", result["source_kind"])
    print("Source type:", result["source_type"])
    print("Topic:", result["topic"])
    print("Title:", result["title"])
    print("Global chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:900])

Query:

Ubuntu server has low disk space, systemd services are restarting,
journal logs are growing, and apt cache may be consuming space.
What should be inspected first?


Search results: 10
Distance: 0.1862
Source kind: knowledge_document
Source type: runbook
Topic: disk_pressure
Title: Ubuntu Disk Pressure and Journal Cleanup Runbook
Global chunk: 61
```json { "document_number": 1, "document_type": "runbook", "topic": "disk_pressure", "title": "Ubuntu Disk Pressure and Journal Cleanup Runbook", "focus": "journalctl disk usage, apt cache cleanup, log rotation, safe disk investigation" } ``` --- ### **1. Problem Overview** This runbook addresses alerts related to disk pressure on Ubuntu servers, with a specific focus on issues originating from log accumulation, particularly `systemd-journald`. When a filesystem, especially the root partition (`/`) or `/var`, approaches full capacity, system stability is compromised. Symptoms manifest as: * Application failures with "No space left on d

In [28]:
def build_ubuntu_rag_context(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Chunk ID: {result['chunk_id']}",
                    f"Source kind: {result['source_kind']}",
                    f"Source type: {result['source_type']}",
                    f"Title: {result['title']}",
                    f"Topic: {result['topic']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [29]:
def audit_ubuntu_issue(
    question: str,
    *,
    top_k: int = 12,
) -> dict:
    results = search_ubuntu_chunks(
        question,
        top_k=top_k,
    )

    context = build_ubuntu_rag_context(results)

    prompt = f"""
You are a senior Ubuntu SRE and security-aware log auditor.

Analyze the issue using only the retrieved log chunks and knowledge chunks.

Question:
{question}

Retrieved chunks:
{context}

Return a practical audit report with:
1. Executive summary
2. Most likely primary issue
3. Evidence from log chunks
4. Relevant knowledge-base guidance
5. Operational risk level: low, medium, high, or critical
6. First diagnostic commands
7. Safe mitigation plan
8. Commands to avoid
9. Escalation criteria
10. Sources used, referencing SOURCE numbers

Rules:
- Stay grounded in retrieved chunks.
- Do not invent evidence.
- Prefer non-destructive diagnostics.
- Do not include destructive commands.
- If evidence is insufficient, say what is missing.
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "question": question,
        "answer": response.text,
        "search_results": results,
        "rag_context": context,
    }

In [30]:
audit_question = """
The Ubuntu worker server is unstable. Services restart repeatedly,
disk usage is close to full, journal logs appear large, and there are apt cache warnings.
Audit the issue and recommend safe diagnostics and mitigation.
"""

ubuntu_audit_result = audit_ubuntu_issue(
    audit_question,
    top_k=12,
)

print("AUDIT QUESTION:")
print(ubuntu_audit_result["question"])

print("\nAUDIT REPORT:")
print(ubuntu_audit_result["answer"])

AUDIT QUESTION:

The Ubuntu worker server is unstable. Services restart repeatedly,
disk usage is close to full, journal logs appear large, and there are apt cache warnings.
Audit the issue and recommend safe diagnostics and mitigation.


AUDIT REPORT:
Here is an audit report for the unstable Ubuntu worker server:

### 1. Executive Summary

The Ubuntu worker server `ubuntu-worker-01` is experiencing critical instability due to severe disk pressure. Multiple services, including Kubernetes Kubelet, systemd-journald, apt, and snapd, are failing to write data, leading to repeated restarts and service unavailability. The root cause is a "No space left on device" condition on the primary filesystem, exacerbated by large journal logs and apt cache issues.

### 2. Most Likely Primary Issue

The primary issue is a **critical "No space left on device" condition** on the root filesystem (likely `/` or `/var`), leading to cascading service failures and instability. This is directly causing service

In [31]:
def generate_cautious_bash_script(audit_result: dict) -> dict:
    used_chunk_ids = [
        result["chunk_id"]
        for result in audit_result["search_results"]
    ]

    used_sources = sorted(
        {
            f"{result['source_kind']} | {result['title']}"
            for result in audit_result["search_results"]
        }
    )

    prompt = f"""
You are a senior Linux SRE.

Generate a cautious Bash helper script based only on this audit result and retrieved context.

Audit question:
{audit_result["question"]}

Audit answer:
{audit_result["answer"]}

Retrieved context:
{audit_result["rag_context"]}

Return only valid JSON:

{{
  "risk_level": "low | medium | high | critical",
  "primary_issue": "short issue name",
  "root_cause_hypothesis": "short hypothesis",
  "bash_script": "complete bash script as a single string"
}}

Script rules:
- The script must be diagnostic-first.
- It may print disk usage, journal usage, large directories, failed systemd units, and recent errors.
- It may suggest safe commands as echo statements.
- It must not execute destructive cleanup automatically.
- It must not use rm -rf.
- It must not modify firewall, users, ssh keys, or package sources.
- It must include comments explaining what each section does.
- It must start with: #!/usr/bin/env bash
- It must include: set -euo pipefail
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    script_data = extract_json_object(response.text)

    return {
        "audit_id": str(uuid4()),
        "question": audit_result["question"],
        "risk_level": script_data.get("risk_level"),
        "primary_issue": script_data.get("primary_issue"),
        "root_cause_hypothesis": script_data.get("root_cause_hypothesis"),
        "audit_report": audit_result["answer"],
        "bash_script": script_data.get("bash_script"),
        "used_chunk_ids": used_chunk_ids,
        "used_sources": used_sources,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }


audit_record = generate_cautious_bash_script(ubuntu_audit_result)

print("Risk level:", audit_record["risk_level"])
print("Primary issue:", audit_record["primary_issue"])
print("Root cause hypothesis:", audit_record["root_cause_hypothesis"])
print("\nBash script:")
print(audit_record["bash_script"])

Risk level: high
Primary issue: Critical Disk Pressure
Root cause hypothesis: The root filesystem is full, primarily due to large systemd journal logs and an uncleaned apt cache, which is causing cascading service failures including Kubelet.

Bash script:
#!/usr/bin/env bash
# Cautious SRE Diagnostic Helper for Ubuntu Disk Pressure
# This script is READ-ONLY. It will not modify the system.
# It will diagnose the state and suggest safe, copy-pasteable commands based on the audit findings.

set -euo pipefail

# --- Script Preamble and Safety Check ---
echo "--- [SRE Diagnostic Helper - READ-ONLY MODE] ---"
echo "This script will perform a series of safe, read-only checks to diagnose disk pressure."
echo "It is based on audit findings pointing to journal and apt cache issues."
echo "No changes will be made to the system by this script."

if [[ $EUID -ne 0 ]]; then
  echo "
⚠️ Warning: Not running as root. Some commands requiring 'sudo' may fail."
fi

# --- Phase 1: System Diagnostics (Rea

In [32]:
audit_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

audit_report_blob_name = (
    f"{GCS_AUDIT_PREFIX}/"
    f"{audit_record['audit_id']}/"
    f"audit_report_{audit_timestamp}.md"
)

script_blob_name = (
    f"{GCS_SCRIPT_PREFIX}/"
    f"{audit_record['audit_id']}/"
    f"mitigation_helper_{audit_timestamp}.sh"
)

audit_gcs_uri = upload_text_to_gcs(
    audit_record["audit_report"],
    blob_name=audit_report_blob_name,
    content_type="text/markdown",
)

script_gcs_uri = upload_text_to_gcs(
    audit_record["bash_script"],
    blob_name=script_blob_name,
    content_type="text/x-shellscript",
)

audit_record = {
    **audit_record,
    "audit_gcs_uri": audit_gcs_uri,
    "script_gcs_uri": script_gcs_uri,
}

print("Audit report saved to:")
print(audit_gcs_uri)

print("\nBash helper script saved to:")
print(script_gcs_uri)

Audit report saved to:
gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/audits/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/audit_report_20260702_154807.md

Bash helper script saved to:
gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/scripts/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/mitigation_helper_20260702_154807.sh


In [33]:
bigquery_client.query(
    f"TRUNCATE TABLE `{audit_table_ref}`",
    location=dataset.location,
).result()

audit_errors = bigquery_client.insert_rows_json(
    audit_table_ref,
    [audit_record],
)

if audit_errors:
    print("Audit insert errors:")
    print(audit_errors)
else:
    print("Inserted audit rows: 1")

Inserted audit rows: 1


In [34]:
test_questions = [
    "SSH auth log shows many failed password attempts and invalid users. What is the risk?",
    "dmesg reports PCIe device reset and driver timeout. What should be checked?",
    "systemd service keeps restarting after config change and port binding error.",
    "NetworkManager reports DNS timeout and interface link flaps.",
]

for question in test_questions:
    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    results = search_ubuntu_chunks(
        question,
        top_k=5,
    )

    for result in results:
        print(
            "distance:",
            round(result["distance"], 4),
            "| kind:",
            result["source_kind"],
            "| type:",
            result["source_type"],
            "| title:",
            result["title"],
            "| chunk:",
            result["global_chunk_number"],
        )


QUESTION:
SSH auth log shows many failed password attempts and invalid users. What is the risk?
distance: 0.2235 | kind: knowledge_document | type: runbook | title: SSH Authentication Anomaly Investigation Runbook | chunk: 87
distance: 0.2502 | kind: knowledge_document | type: runbook | title: SSH Authentication Anomaly Investigation Runbook | chunk: 88
distance: 0.2546 | kind: log_file | type: authlog | title: ubuntu-api-03 authlog log | chunk: 28
distance: 0.2594 | kind: knowledge_document | type: runbook | title: SSH Authentication Anomaly Investigation Runbook | chunk: 93
distance: 0.2598 | kind: knowledge_document | type: runbook | title: SSH Authentication Anomaly Investigation Runbook | chunk: 90

QUESTION:
dmesg reports PCIe device reset and driver timeout. What should be checked?
distance: 0.173 | kind: knowledge_document | type: runbook | title: Kernel Driver Warning and Device Reset Runbook | chunk: 105
distance: 0.2196 | kind: log_file | type: dmesg | title: ubuntu-gpu-nod

In [35]:
sql = f"""
SELECT
  source_kind,
  source_type,
  topic,
  COUNT(*) AS chunk_count,
  ROUND(AVG(chunk_char_count), 2) AS avg_chunk_chars,
  MIN(chunk_char_count) AS min_chunk_chars,
  MAX(chunk_char_count) AS max_chunk_chars
FROM `{chunk_table_ref}`
GROUP BY
  source_kind,
  source_type,
  topic
ORDER BY
  source_kind,
  chunk_count DESC
"""

chunk_analytics_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

chunk_analytics_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,source_kind,source_type,topic,chunk_count,avg_chunk_chars,min_chunk_chars,max_chunk_chars
0,knowledge_document,runbook,ssh_auth_anomaly,16,1014.63,347,1200
1,knowledge_document,runbook,systemd_failures,14,1058.36,712,1199
2,knowledge_document,runbook,kernel_driver_warnings,13,1051.77,882,1199
3,knowledge_document,runbook,disk_pressure,12,1073.75,437,1200
4,knowledge_document,runbook,network_instability,11,1109.45,735,1200
5,log_file,syslog,disk pressure and journal growth causing servi...,15,967.33,375,1200
6,log_file,network,network instability with DNS failures and inte...,14,1080.00,570,1200
7,log_file,authlog,SSH authentication anomaly with repeated faile...,11,1160.73,970,1200
8,log_file,dmesg,kernel driver warnings and intermittent device...,10,1108.60,790,1200
9,log_file,systemd,database service restart loop after configurat...,10,1122.80,1013,1199


In [36]:
sql = f"""
SELECT
  audit_id,
  risk_level,
  primary_issue,
  root_cause_hypothesis,
  audit_gcs_uri,
  script_gcs_uri,
  created_at
FROM `{audit_table_ref}`
ORDER BY created_at DESC
"""

audit_overview_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

audit_overview_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,audit_id,risk_level,primary_issue,root_cause_hypothesis,audit_gcs_uri,script_gcs_uri,created_at
0,822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8,high,Critical Disk Pressure,"The root filesystem is full, primarily due to ...",gs://leafy-guide-497515-m4-vector-assets/ubunt...,gs://leafy-guide-497515-m4-vector-assets/ubunt...,2026-07-02 15:46:30.535665+00:00


In [37]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "64_w_google_cloud_ubuntu_log_auditor_rag.ipynb",
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "log_table": log_table_ref,
        "document_table": document_table_ref,
        "chunk_table": chunk_table_ref,
        "audit_table": audit_table_ref,
    },
    "cloud_storage": {
        "log_prefix": f"gs://{BUCKET_NAME}/{GCS_LOG_PREFIX}/{timestamp}",
        "audit_gcs_uri": audit_gcs_uri,
        "script_gcs_uri": script_gcs_uri,
        "local_files_saved": False,
    },
    "log_count": len(generated_logs),
    "knowledge_document_count": len(knowledge_documents),
    "chunk_count": len(chunk_rows),
    "audit_record": audit_record,
    "chunk_analytics": chunk_analytics_df.to_dict(orient="records"),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{audit_record['audit_id']}/"
    f"summary_{audit_timestamp}.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/summaries/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/summary_20260702_154807.json


In [38]:
print("Ubuntu Log Auditor RAG notebook completed.")
print("=" * 100)

print("Generated logs:", len(generated_logs))
print("Knowledge documents:", len(knowledge_documents))
print("Chunks:", len(chunk_rows))

print("\nBigQuery tables:")
print("-", log_table_ref)
print("-", document_table_ref)
print("-", chunk_table_ref)
print("-", audit_table_ref)

print("\nGCS outputs:")
print("Logs prefix:", f"gs://{BUCKET_NAME}/{GCS_LOG_PREFIX}/{timestamp}")
print("Audit report:", audit_gcs_uri)
print("Bash helper script:", script_gcs_uri)
print("Summary:", summary_gcs_uri)

print("\nAudit result:")
print("Risk level:", audit_record["risk_level"])
print("Primary issue:", audit_record["primary_issue"])
print("Root cause hypothesis:", audit_record["root_cause_hypothesis"])

print("\nAudit report preview:")
print(audit_record["audit_report"][:1600])

print("\nBash script preview:")
print(audit_record["bash_script"][:1600])

print("\nNo local files were saved.")

Ubuntu Log Auditor RAG notebook completed.
Generated logs: 5
Knowledge documents: 5
Chunks: 126

BigQuery tables:
- leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_files
- leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_knowledge_documents
- leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_log_and_knowledge_chunks
- leafy-guide-497515-m4.ubuntu_log_auditor_rag.ubuntu_audit_reports

GCS outputs:
Logs prefix: gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/logs/20260702_151254
Audit report: gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/audits/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/audit_report_20260702_154807.md
Bash helper script: gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/scripts/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/mitigation_helper_20260702_154807.sh
Summary: gs://leafy-guide-497515-m4-vector-assets/ubuntu-log-auditor/summaries/822cafc5-c8a2-47d0-b3c3-bed2e7cf74b8/summary_20260702_154807.json

Audit result:
Risk level: high
Prim